# 143. Reorder List
**Difficulty:** 🟡 Medium · **Topic:** Linked List · **LeetCode:** https://leetcode.com/problems/reorder-list/

## 💡 Concepts

**Core concept(s):** Three classic moves — **find the middle**, **reverse the second half**, **weave** the halves together.

**Why it applies here:** The target order L0, Ln, L1, Ln-1, … alternates from the front and the back. Splitting at the middle, reversing the back half, then zipping the two halves produces exactly that with O(1) extra space.

**Key intuition:** Fold the list: cut in the middle, flip the back half, then interleave front and back.

---

### 📚 What is a Linked List?
A **linked list** is a chain of nodes; each node holds a value and a pointer to the **next** node. Unlike an array there is no index — you can only walk forward from the head.
- **In Python:** a small `ListNode` class with `.val` and `.next`.

### 📚 Fast & Slow Pointers
Two pointers moving at different speeds: **slow** one step, **fast** two. They meet inside a loop (cycle detection) and the slow one lands on the middle when fast reaches the end.
- **Complexity:** one pass, **O(1)** extra space.

### 📚 Pointers & the Dummy Node
Linked-list code moves **pointers** (references to nodes). A **dummy** node placed before the head removes annoying "is this the first node?" special cases — you build off `dummy.next` and return it at the end.

---

**Prerequisite knowledge:**
- Find middle (fast/slow).
- Reverse a list.
- Splicing two lists together.

## 📝 Problem

Reorder `L0->L1->...->Ln` into `L0->Ln->L1->Ln-1->...` in place.

**Example**
```
1->2->3->4  ->  1->4->2->3
```

> Two approaches: array of nodes `O(n)` space, and in-place `O(1)` space (find-middle + reverse + merge).

In [ ]:
from typing import Optional, List

class ListNode:
    """A node in a singly linked list: a value plus a link to the next node."""
    def __init__(self, val=0, next=None):
        self.val = val                     # the value stored at this node
        self.next = next                   # link to the next node (None at the end)

def build_list(vals):
    """Turn a Python list into a linked list; return its head."""
    dummy = ListNode(); cur = dummy        # dummy node avoids special-casing the first node
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head):
    """Turn a linked list back into a Python list (handy for printing / assertions)."""
    out = []
    while head:
        out.append(head.val); head = head.next
    return out

### Approach 1 — Array of Nodes (worst on memory)

**Idea:** Put all nodes in an array, then rewire with two pointers from both ends.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
def reorder_array(head):
    if not head:
        return head
    nodes = []; cur = head
    while cur:                             # collect all nodes into an array for easy indexing
        nodes.append(cur); cur = cur.next
    i, j = 0, len(nodes) - 1               # two pointers from both ends
    while i < j:
        nodes[i].next = nodes[j]; i += 1   # link front node -> back node
        if i == j:
            break
        nodes[j].next = nodes[i]; j -= 1   # link that back node -> next front node
    nodes[i].next = None                   # terminate the list at the middle
    return head

### Approach 2 — Middle + Reverse + Merge (optimal)

**Idea:** Fast/slow to find the middle, reverse the second half, then interleave the two halves.

**Time:** `O(n)`. **Space:** `O(1)`.

In [ ]:
def reorder_inplace(head):
    if not head or not head.next:
        return head
    slow = fast = head                     # 1) find the middle with fast/slow pointers
    while fast.next and fast.next.next:
        slow = slow.next; fast = fast.next.next
    second = slow.next; slow.next = None    # split into two halves
    prev = None                            # 2) reverse the second half
    while second:
        nxt = second.next; second.next = prev; prev = second; second = nxt
    first, second = head, prev             # 3) weave the two halves together
    while second:
        n1, n2 = first.next, second.next   # save the next nodes on both sides
        first.next = second; second.next = n1  # interleave: first -> second -> (old first.next)
        first, second = n1, n2             # advance into both halves
    return head

In [ ]:
# Correctness check
tests = [([1,2,3,4],[1,4,2,3]), ([1,2,3,4,5],[1,5,2,4,3]), ([1],[1]), ([1,2],[1,2])]
for vals, exp in tests:
    assert to_list(reorder_array(build_list(vals))) == exp
    assert to_list(reorder_inplace(build_list(vals))) == exp
    print(vals, "->", exp)
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio when `n` → `2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_list(list(range(n))),)
solutions = {
    "array    O(n) space O(n)": reorder_array,
    "in-place O(n) space O(1)": reorder_inplace,
}
sizes = [20000, 40000, 80000, 160000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Compose three sub-routines:** find middle + reverse + merge — each a reusable list skill.
- **Signal:** "reorder / fold / rearrange a list by position".
- **Related problems:** Palindrome Linked List, Reverse Linked List, Middle of the Linked List.
- **Common pitfalls:** (1) not cutting the first half's tail (`slow.next = None`); (2) off-by-one picking the middle.